# Défi du jour : construire un système RAG (Retrieval Augmented Generation)

**Objectif du notebook** : construire, étape par étape, un système capable de répondre à des questions en s'appuyant sur un vrai jeu de données Hugging Face (`databricks/databricks-dolly-15k`), grâce à LangChain.

Ce notebook est écrit pour un·e débutant·e : chaque cellule est commentée en français et explique **ce qu'on fait** et **pourquoi**.

⚠️ **Avertissement honnête avant de commencer** (je te le dis franchement, ce n'est pas dans l'énoncé) :
Le sujet demande d'utiliser un modèle de **Question-Answering extractif** (`Intel/dynamic_tinybert`) comme "LLM" dans une `RetrievalQA` avec `chain_type="refine"`. 
Techniquement, ça ne fonctionne pas correctement : un modèle QA extractif répond en extrayant un passage d'un contexte donné (question + contexte en entrée séparés), alors que `RetrievalQA` attend un LLM génératif qui reçoit un **prompt texte unique** et génère du texte en sortie. Le mode `refine` en plus suppose plusieurs appels génératifs successifs, ce qu'un pipeline QA extractif ne sait pas faire.

Donc : je vais suivre l'énoncé à la lettre dans les sections 1 à 7 (pour que tu aies exactement ce qui est demandé), **et** j'ajouterai une section 8 bis avec une version qui fonctionne réellement (un vrai modèle génératif), pour que tu voies la différence et que tu ne rendes pas un devoir qui plante à l'exécution.


## Étape 1 — Installer les bibliothèques nécessaires

On installe tous les outils dont on a besoin :
- **langchain** : orchestre les différentes briques (chargement, découpage, recherche, génération)
- **transformers** : donne accès aux modèles pré-entraînés Hugging Face
- **sentence-transformers** : sert à transformer du texte en vecteurs (embeddings)
- **datasets** : permet de charger des jeux de données Hugging Face
- **faiss-cpu** : moteur de recherche de similarité rapide (la "base de données vectorielle")


In [ ]:
# Installation des bibliothèques (à exécuter une seule fois)
!pip install -q langchain
!pip install -q torch
!pip install -q transformers
!pip install -q sentence-transformers
!pip install -q datasets
!pip install -q faiss-cpu
!pip install -U langchain-community
!pip install -Uq datasets


## Étape 2 — Charger le jeu de données

On utilise `HuggingFaceDatasetLoader` pour charger le dataset `databricks/databricks-dolly-15k`
et transformer chaque ligne en un "document" que LangChain peut manipuler.

On choisit la colonne `context` comme contenu principal du document (c'est le texte que le système va indexer et dans lequel il va chercher les réponses).


In [ ]:
# On importe le loader spécialisé pour les datasets Hugging Face
from langchain.document_loaders import HuggingFaceDatasetLoader

# Nom du dataset sur Hugging Face Hub
dataset_name = "databricks/databricks-dolly-15k"

# Colonne du dataset qui contient le texte que l'on veut indexer
page_content_column = "context"

# Création du loader et chargement des données en documents LangChain
loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()

# On vérifie que le chargement a fonctionné en affichant les 2 premiers documents
print(data[:2])


## Étape 3 — Découper les documents en petits morceaux (chunks)

Les modèles de langage ont une limite de taille de texte qu'ils peuvent traiter d'un coup.
On découpe donc les documents en morceaux plus petits, avec un léger chevauchement (`chunk_overlap`)
pour ne pas couper une information importante en plein milieu.


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# chunk_size : taille maximale d'un morceau de texte (en caractères)
# chunk_overlap : nombre de caractères communs entre deux morceaux consécutifs
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

# On découpe tous les documents chargés à l'étape précédente
docs = text_splitter.split_documents(data)

# Vérification : on regarde le premier morceau obtenu
print(docs[0])


## Étape 4 — Transformer le texte en vecteurs (embeddings)

Un ordinateur ne comprend pas directement le sens d'une phrase : il faut la transformer en une liste de nombres (un **vecteur**) qui capture son sens.
On utilise un modèle de la famille `sentence-transformers`, conçu spécialement pour ça.


In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

# Modèle léger et rapide, très utilisé pour les embeddings de phrases
modelPath = "sentence-transformers/all-MiniLM-l6-v2"

# On force l'utilisation du CPU (pas besoin de carte graphique pour ce modèle)
model_kwargs = {'device': 'cpu'}

# On ne normalise pas les vecteurs (choix par défaut demandé dans l'énoncé)
encode_kwargs = {'normalize_embeddings': False}

embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# Test rapide : on transforme une phrase en vecteur et on affiche les 3 premiers nombres
text = "This is a test document."
query_result = embeddings.embed_query(text)
print(query_result[:3])


## Étape 5 — Créer la base de données vectorielle (FAISS)

FAISS indexe tous les vecteurs créés à l'étape précédente. Il permet ensuite de retrouver,
en quelques millisecondes, les morceaux de texte les plus proches (donc les plus pertinents) d'une question posée.

⚠️ Cette étape peut prendre plusieurs minutes selon la taille du dataset : `databricks-dolly-15k` contient environ 15 000 lignes,
donc si tu veux tester rapidement, réduis `data` à un échantillon avant de le passer au splitter (ex : `data[:500]`).


In [ ]:
from langchain.vectorstores import FAISS

# Construction de l'index vectoriel à partir des morceaux de texte (docs) et du modèle d'embeddings
db = FAISS.from_documents(docs, embeddings)


## Étape 6 — Préparer le modèle de langage (LLM)

C'est le modèle qui va générer la réponse finale à partir des passages retrouvés.
L'énoncé demande d'utiliser `Intel/dynamic_tinybert`, un modèle de **question-answering extractif**.


In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from langchain import HuggingFacePipeline

model_name = "Intel/dynamic_tinybert"

# Chargement du tokenizer et du modèle de question-answering
tokenizer = AutoTokenizer.from_pretrained(model_name, padding=True, truncation=True, max_length=512)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# Création du pipeline Hugging Face de type "question-answering"
qa_pipeline = pipeline(
    "question-answering",
    model=model_name,
    tokenizer=tokenizer,
    return_tensors='pt'
)

# On enveloppe ce pipeline dans un objet compatible LangChain
llm = HuggingFacePipeline(
    pipeline=qa_pipeline,
    model_kwargs={"temperature": 0.7, "max_length": 512},
)


## Étape 7 — Construire la chaîne Retrieval QA

On relie le retriever (qui va chercher les documents pertinents dans FAISS) au LLM
(qui génère la réponse à partir de ces documents).


In [ ]:
from langchain.chains import RetrievalQA

# Le retriever renvoie les 4 morceaux de texte les plus proches de la question
retriever = db.as_retriever(search_kwargs={"k": 4})

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="refine",
    retriever=retriever,
    return_source_documents=False
)


## Étape 8 — Tester le système

On pose une question et on regarde la réponse générée.

⚠️ **À ce stade, il est très probable que cette cellule lève une erreur ou renvoie un résultat incohérent.**
Ce n'est pas une erreur de ta part : c'est la conséquence directe du problème signalé en introduction (pipeline QA extractif utilisé là où LangChain attend un LLM génératif, en mode `refine` qui suppose plusieurs passes de génération).
Si ça plante ici, ne perds pas de temps à chercher un bug dans ton code : va directement à la section 8 bis.


In [ ]:
question = "What is cheesemaking?"

result = qa.run({"query": question})
print(result)


## Étape 8 bis — Version qui fonctionne réellement (bonus, honnête)

Ici on remplace le modèle QA extractif par un vrai modèle **génératif** (`text2text-generation`),
et on utilise `chain_type="stuff"` (plus simple et compatible avec un seul appel au modèle).
C'est la version que je te recommande de comprendre si tu veux réellement voir un RAG fonctionner de bout en bout.


In [ ]:
from transformers import AutoTokenizer as T2TTokenizer, AutoModelForSeq2SeqLM, pipeline as hf_pipeline
from langchain import HuggingFacePipeline as HFPipelineGen
from langchain.chains import RetrievalQA as RetrievalQAGen

gen_model_name = "google/flan-t5-base"  # modèle génératif léger, adapté au CPU

gen_tokenizer = T2TTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(gen_model_name)

text2text_pipeline = hf_pipeline(
    "text2text-generation",
    model=gen_model,
    tokenizer=gen_tokenizer,
    max_length=512
)

llm_gen = HFPipelineGen(pipeline=text2text_pipeline)

qa_gen = RetrievalQAGen.from_chain_type(
    llm=llm_gen,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=False
)

question = "What is cheesemaking?"
result_gen = qa_gen.run({"query": question})
print(result_gen)


## Conclusion

- Les étapes 1 à 7 respectent l'énoncé au mot près.
- L'étape 8 montre le résultat attendu par le sujet — qui risque de ne pas marcher, pour la raison technique expliquée.
- L'étape 8 bis montre une architecture RAG qui fonctionne vraiment, avec un modèle génératif.

Si ton exercice est noté strictement sur la conformité au sujet, garde les sections 1–8 telles quelles.
Si tu veux un système qui marche pour de vrai (par exemple pour une démo), garde la section 8 bis.
